# Cleaned imports and helper functions
# pip install mtcnn tensorflow pillow matplotlib numpy scikit-learn
import os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import tensorflow as tf
from mtcnn import MTCNN
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.preprocessing.image import ImageDataGenerator
detector = MTCNN()

def detect_faces_pil(pil_img):
    arr = np.asarray(pil_img)
    faces = detector.detect_faces(arr)
    boxes = []
    for f in faces:
        x,y,w,h = f['box']
        x,y = max(0,x), max(0,y)
        boxes.append((x,y,w,h))
    return boxes

def crop_face(pil_img, box, size=(224,224), margin=0.2):
    w_img,h_img = pil_img.size
    x,y,w,h = box
    dx = int(w * margin)
    dy = int(h * margin)
    x1 = max(0, x - dx)
    y1 = max(0, y - dy)
    x2 = min(w_img, x + w + dx)
    y2 = min(h_img, y + h + dx)
    face = pil_img.crop((x1,y1,x2,y2)).resize(size)
    return face

def build_finetune_model(input_shape=(224,224,3), lr=1e-4):
    base = EfficientNetB0(weights='imagenet', include_top=False, input_shape=input_shape, pooling='avg')
    for layer in base.layers:
        layer.trainable = True
    x = base.output
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(1, activation='sigmoid')(x)
    model = models.Model(inputs=base.input, outputs=out)
    model.compile(optimizer=tf.keras.optimizers.Adam(lr), loss='binary_crossentropy', metrics=['accuracy'])
    return model

def prepare_generators(data_dir, img_size=(224,224), batch_size=16, augmentation=True):
    if augmentation:
        datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2, rotation_range=20, width_shift_range=0.15, height_shift_range=0.15, shear_range=0.1, zoom_range=0.15, horizontal_flip=True, fill_mode='nearest')
    else:
        datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)
    train_gen = datagen.flow_from_directory(data_dir, target_size=img_size, batch_size=batch_size, class_mode='binary', subset='training')
    val_gen = datagen.flow_from_directory(data_dir, target_size=img_size, batch_size=batch_size, class_mode='binary', subset='validation')
    return train_gen, val_gen

from collections import Counter
def train(data_dir, model_save_path='models/efficientnet_face.h5', epochs=10, batch_size=16, augmentation=True):
    os.makedirs(os.path.dirname(model_save_path), exist_ok=True)
    train_gen, val_gen = prepare_generators(data_dir, img_size=(224,224), batch_size=batch_size, augmentation=augmentation)
    counter = Counter(train_gen.classes)
    total = sum(counter.values())
    class_weight = {cls: total/(len(counter)*count) for cls, count in counter.items()}
    print('class_weight:', class_weight)
    model = build_finetune_model()
    model.fit(train_gen, validation_data=val_gen, epochs=epochs, class_weight=class_weight)
    model.save(model_save_path)
    return model

def predict_image(img_path, model, threshold=0.5):
    pil = Image.open(img_path).convert('RGB')
    boxes = detect_faces_pil(pil)
    results = []
    for box in boxes:
        face = crop_face(pil, box)
        arr = np.array(face)/255.0
        pred = float(model.predict(np.expand_dims(arr,0))[0][0])
        label = 'me' if pred >= threshold else 'unknown'
        results.append({'box':box, 'score':pred, 'label':label})
    return results